# The Eval Your Downstream System Actually Needs

*Score JSON schema compliance, instruction adherence, and exact-match rules in DeepEval without paying for a judge call.*

**Read this if**

- Your LLM output feeds another system rather than a person, and that system has opinions about formats.
- Your eval bill looks high for checks that are really just assertions.
- You want to know which of your criteria can be demoted to code, and what that does to the bill and to flakiness.

The previous post left Halcyon with two house rules written as metrics. Both run on live answers, and one of them is a decision tree. Deciding those two questions cost 500 judge calls and $\$1.0388$. The tree spent three calls per case to be right about two more verdicts out of fifty. That price was fair for those rules. "Does this answer recommend a deprecated scheme" has no regular expression. Most of what a ticketing system needs checked does.

Halcyon is a payments API vendor with about two hundred staff. Its customers are developers integrating card payments. Every inbound support ticket is a paragraph a developer wrote. Halcyon's ticketing system does not want a paragraph. It wants a record with seven fields. The ticket id. The account id. The API version the developer named. The endpoint the ticket is about. The error code they saw. A severity, P1 to P3, under Halcyon's support rules. And whether the ticket reports that money moved wrongly. A sync job loads those records overnight. The support engineer on the morning shift reads the queue sorted by them.

In August a record reached the sync job with an account id in capitals, and a dash where the format has an underscore. The record was valid JSON. Every field was present. The account lookup failed at three in the morning, the row was skipped, and the ticket never reached the morning queue. It was a refund that had not arrived, the kind of report Halcyon promises to answer within an hour. The developer wrote again two days later. No check had looked at the account id. The check that would have caught it is one line of code.

## Which checks need a judge

A judge is a second language model that reads an output and returns a score with a reason. A metric is the code around the judge. It decides what the judge is asked, how many times, and how the reply becomes a number. Some metrics have no judge inside them at all. They run a comparison or a pattern and return 1 or 0. We call those deterministic. The same input gives the same score on every run, because nothing inside them can vary. A test case is the record a metric reads. Here it holds the ticket and the JSON the model wrote from it. Sometimes it also holds the value we know to be right. A metric that needs that value is reference-based. One that reads only what the system produced is referenceless. The threshold is the minimum score that counts as a pass. Every DeepEval score is higher-is-better, so the threshold is always a floor.

A unit test is the analogy. `assert severity in ("P1", "P2", "P3")` needs no reviewer. Nobody asks a senior engineer to eyeball an enum. Code review is for the questions an assertion cannot ask. Is this the right severity for this ticket? Is that 500 Halcyon's error, or the developer's own? The analogy holds for the split and stops at the reviewer. A reviewer asked the same question twice gives the same answer. A judge, as we will see, does not.

Here is the plan. We build Halcyon's third application, a Ticket Field Extractor, as one LangGraph node. A node is a single step in a graph, and this one turns a ticket into a record. Then we score the same records four ways with DeepEval's own metrics. A schema check, a pattern check and an exact-match check, all at zero judge calls. Then an instruction-compliance check that costs two. Then we write one custom metric that puts the code checks in front of a single judged question. That is the shape the previous post built as a tree. We run every metric three times and report two things per metric. Whether it returned the same score every time, and whether it was right. We close on a field that looks like a keyword match and is not.

The notebook reads `GEMINI_API_KEY` from the environment. A full run costs about seventy cents, and the exact bill is printed at the end.

The shared setup cell from the first post in the series, re-created unchanged. It prints the corpus fingerprint and stops if a page or a ticket has drifted.

In [1]:
# Shared Halcyon scaffolding, built in post 0 and re-created verbatim in every later post.
# !uv pip install deepeval==4.2.1 langgraph==1.2.11 langchain-core==1.6.2 langchain-google-genai==4.4.0 google-genai==2.22.0
import os, re, json, time, hashlib, logging, statistics
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from deepeval.dataset import Golden

logging.getLogger("google_genai.models").setLevel(logging.ERROR)   # silence an SDK advisory about tool calling

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("Set GEMINI_API_KEY in the environment before running this notebook.")

JUDGE_MODEL = "gemini-3.6-flash"      # the judge every later post uses; this post never calls it
APP_MODEL = "gemini-3.6-flash"        # the model inside the assistant under test
MAX_CONCURRENCY = 4                   # model calls in flight at any moment
PRICE_IN, PRICE_OUT = 0.75 / 1e6, 3.75 / 1e6   # USD per token, gemini-3.6-flash standard tier, 2026 rate
T0 = time.time()
USAGE = []                            # one usage_metadata dict per model call, for the cost ledger


def with_retry(fn, *args, attempts=5, base_delay=2.0):
    """Call fn(*args); on a 429 or 503 wait 2, 4, 8, 16 seconds and try again."""
    for attempt in range(attempts):
        try:
            return fn(*args)
        except Exception as exc:
            transient = any(code in str(exc) for code in ("429", "503", "RESOURCE_EXHAUSTED"))
            if not transient or attempt == attempts - 1:
                raise
            time.sleep(base_delay * 2 ** attempt)


# No temperature, top_p or top_k: Google lists them as deprecated sampling parameters to strip from requests.
llm = ChatGoogleGenerativeAI(model=APP_MODEL, google_api_key=os.environ["GEMINI_API_KEY"], max_retries=0)

DOCS = [dict(zip(('slug', 'title', 'status', 'updated', 'body'), row)) for row in [
    ("quickstart", "Quickstart", "current", "2026-03-01", "Create an account, then generate a test API key from the dashboard; test keys start with sk_test_ and live keys with sk_live_. Install the SDK with pip install halcyon (the current SDK is 3.x). Create your first charge with POST /v1/charges, passing amount in minor units (an integer: 1999 means 19.99), a lowercase ISO 4217 currency such as usd or eur, and a source token from Halcyon.js. Send the Halcyon-Version header with the API version you tested against so later changes do not reach you until you opt in. Use test card 4242 4242 4242 4242 for a successful payment and 4000 0000 0000 0002 for a decline."),
    ("api-conventions", "API conventions", "current", "2026-03-01", "The API is served over HTTPS at https://api.halcyon.dev and is date-versioned: send Halcyon-Version: 2026-03-01 (or an earlier version) with each request; requests without the header use the account's default version. Amounts are integers in minor units and currencies are lowercase ISO 4217 codes. Every POST to a money-moving endpoint (charges, refunds, payouts) must include an Idempotency-Key header, a client-generated unique string; a UUID v4 is recommended. Replaying a request with the same key within 24 hours returns the original response instead of moving money again. Requests without the header are accepted for backward compatibility, so a retried POST without one creates a second charge. Keys are scoped to the account and to the endpoint."),
    ("authentication", "Authentication", "current", "2026-01-15", "Authenticate every request with an Authorization: Bearer header carrying your secret key. Test-mode keys (sk_test_) never move real money; live-mode keys (sk_live_) do. Restricted keys can be limited to read-only access or to specific resources. Rotate a key from the dashboard: the new key is active immediately and the old key keeps working for 24 hours so deployments can roll over without downtime; revoke it sooner from the same screen if it was exposed. A missing or invalid key returns HTTP 401 with type authentication_error. Never send secret keys from browsers or mobile apps; use Halcyon.js publishable keys (pk_) there."),
    ("errors", "Errors", "current", "2026-03-01", "Errors return a JSON body with type, code, message and request_id. Types map to HTTP status: invalid_request_error (400), authentication_error (401), card_error (402), rate_limit_error (429) and api_error (500 to 504). card_error codes include card_declined, insufficient_funds, expired_card and incorrect_cvc; show the cardholder a generic message and let them try another card. Requests that return 5xx may have partially completed: retry them with exponential backoff, and quote the request_id when you contact support. Do not retry 4xx errors other than 429."),
    ("rate-limits", "Rate limits", "current", "2025-11-01", "Live mode allows 100 requests per second per account with short bursts to 200; test mode allows 25 requests per second. Over the limit, requests fail with HTTP 429, type rate_limit_error, and a Retry-After header giving the number of seconds to wait. Back off exponentially with jitter rather than retrying in a tight loop, and spread bulk work such as migrations or invoice runs over time or through a queue. Webhook deliveries to your endpoint do not count against the limit. Contact support with your expected peak if you need a higher limit."),
    ("charges-create", "Create a charge", "current", "2026-03-01", "POST /v1/charges creates a charge. Required parameters: amount (integer, minor units), currency (lowercase ISO 4217) and source (a token from Halcyon.js or a saved payment method id starting pm_). Optional: description, metadata (up to 20 key-value pairs), and capture, which defaults to true; pass capture=false to authorize only, then capture within 7 days with POST /v1/charges/{id}/capture. The response is a charge object with id (ch_), status (succeeded, pending or failed), amount, currency, amount_refunded and failure_code. Card declines return HTTP 402 with a card_error body; a charge that could not be attempted at all returns 400."),
    ("refunds", "Refunds", "current", "2026-02-10", "POST /v1/refunds with charge (a ch_ id) refunds a charge in full; pass amount to refund part of it. A charge can be refunded several times until the refunded total reaches the original amount; a request beyond the remaining balance returns 400 with code amount_exceeds_refundable. Refunds of a pending charge are rejected until the charge succeeds. The refund object has id (re_), status (pending, succeeded or failed) and amount. Refunds settle to the cardholder in 5 to 10 business days depending on the issuing bank; status succeeded means Halcyon has released the funds, not that the cardholder has seen them yet."),
    ("webhooks-overview", "Webhooks", "current", "2026-03-01", "Halcyon notifies your endpoint about events by POSTing a JSON event object with id (evt_), type, created and data. Event types include charge.succeeded, charge.failed, refund.succeeded and dispute.opened. Your endpoint must return a 2xx within 10 seconds; otherwise the delivery is retried with exponential backoff (1 minute, 5 minutes, 30 minutes, 2 hours, then every 6 hours) for up to 72 hours, after which the event is marked failed and shown in the dashboard. Delivery is at-least-once: the same event can arrive more than once and events can arrive out of order, so store the event id and ignore repeats. Always verify the signature header before trusting an event; see the webhook signatures pages."),
    ("webhook-signatures-v1", "Webhook signatures (v1)", "deprecated", "2026-03-01", "Each webhook request carries an X-Halcyon-Signature header containing a hex-encoded HMAC-SHA1 of the raw request body, keyed with your endpoint's signing secret (whsec_). To verify, compute HMAC-SHA1 over the exact bytes you received, before any JSON parsing, and compare with the header using a constant-time comparison. Reject the event if the values differ. Do not re-serialize the JSON before hashing, since key ordering and whitespace change the digest. This scheme was deprecated on 2026-03-01 in favour of webhook-signatures-v2, which uses HMAC-SHA256 with a timestamped Halcyon-Signature header; it remains in service for accounts pinned to API versions before 2026-03-01."),
    ("webhook-signatures-v2", "Webhook signatures (v2)", "current", "2026-03-01", "Each webhook request carries a Halcyon-Signature header of the form t=<unix timestamp>,v2=<hex digest>. The signed payload is the timestamp, a period, and the raw request body. To verify, compute HMAC-SHA256 of that payload with your endpoint's signing secret (whsec_), compare with the v2 value using a constant-time comparison, and reject the event if the timestamp is more than 300 seconds old, which defeats replay of captured requests. During a signing secret rotation the header carries two v2 values for 24 hours; accept the event if either matches. This is the default scheme for API version 2026-03-01 and later, and the only scheme SDK 3.x verifies."),
    ("changelog-2026-03", "Changelog: API version 2026-03-01", "current", "2026-03-01", "API version 2026-03-01. Webhook signatures v2 (HMAC-SHA256, timestamped Halcyon-Signature header) are now the default; v1 (HMAC-SHA1, X-Halcyon-Signature) is deprecated and is sent only to accounts pinned to earlier versions. Halcyon SDK 3.0 ships with v2 verification helpers and no longer verifies v1 headers; SDK 2.x continues to verify v1 only. The Idempotency-Key replay window is extended from 1 hour to 24 hours, and reusing a key with a different request body now returns 400 with code idempotency_key_reused instead of silently returning the earlier response. Request and response shapes for charges and refunds are unchanged."),
    ("support-sla", "Support and SLA", "current", "2026-01-15", "Support is available to all accounts through the dashboard and support@halcyon.dev. Tickets are triaged into three priorities. P1: any report that funds have been affected, including duplicate or double charges, charges of the wrong amount, and refunds or payouts that did not arrive, regardless of how the report is worded or whether the ticket is mainly about something else; first response within 1 hour, 24 hours a day. P2: a live-mode integration that is failing, such as authentication errors, webhook deliveries or signature verification failing, or unexpected 4xx or 5xx responses from live endpoints; first response within 4 business hours. P3: how-to questions, documentation gaps, feature requests and anything in test mode; first response within 1 business day."),
]]

TICKETS = [dict(zip(('id', 'submitted', 'subject', 'body'), row)) for row in [
    ("T-1001", "2026-08-03", "Which signature header am I supposed to verify?", "We're integrating webhooks and every event we receive has an X-Halcyon-Signature header but no Halcyon-Signature header, so the verify() helper in SDK 3.1 throws 'no v2 signature present'. The webhook overview says to verify signatures and links to two pages. Our account was created in 2025 and I think we're pinned to 2025-11-01. Is v1 still supported, do we need to move to v2, and how do we change the API version without breaking our existing charge flow? This is blocking our launch on the live account."),
    ("T-1002", "2026-08-04", "429s in test mode when our integration suite runs", "Our CI runs about 40 requests per second against test mode at peak and we get bursts of rate_limit_error with Retry-After: 1. Live mode is fine. Is the test-mode limit documented anywhere, and can it be raised for CI? We can add a throttle but it makes the suite roughly three times slower."),
    ("T-1003", "2026-08-05", "Retry storm after our webhook endpoint returned 500s", "On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since then we've received roughly 4,000 redeliveries, some for events from days ago, and they are still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just absorb them until the 72 hours are up? Probably unrelated, but while going through the logs we found a few orders from that afternoon that were charged twice, two charge ids each; we assume our order service retried the charge call on timeout and we'll refund them on our side. Mainly we need the retries to stop."),
    ("T-1004", "2026-08-06", "Getting 400 idempotency_key_reused on retries", "Since we moved to API version 2026-03-01 last week, our retry path fails with 400 idempotency_key_reused. We generate one Idempotency-Key per order and reuse it if the first attempt fails, which used to work. Looking closer, our retry adds a metadata.attempt field to the body. Is the body compared exactly? What is the recommended pattern for retries where the body legitimately changes? This is live traffic; failed retries currently land in a dead-letter queue and someone replays them by hand."),
    ("T-1005", "2026-08-07", "Partial refund returns amount_exceeds_refundable", "A customer paid 120.00 EUR and we refunded 80.00 EUR last week. Refunding the remaining 40.00 EUR today returns 400 amount_exceeds_refundable. It turns out our cancellation flow had already issued a separate 40.00 EUR refund that nobody noticed, so the error is correct. What we actually need: is there a field on the charge that tells us the remaining refundable balance, so we can check before calling refunds? And is there any per-month cap on refunds?"),
    ("T-1006", "2026-08-10", "SDK 3.0 upgrade broke webhook verification, rolled back", "We upgraded from SDK 2.9 to 3.0 on Monday. From the first deploy every webhook failed verification with 'no v2 signature present' and our fulfilment queue stalled. We rolled back to 2.9 after about 25 minutes and verification works again. Our account is on API version 2025-11-01. What exactly changed in 3.0, and what is the migration order: SDK first or API version first? One side effect: while the queue was stalled a colleague re-ran the stuck jobs by hand and two customers were charged a second time; we've refunded one and are chasing the other. The verification question is the one we need answered before we try again."),
    ("T-1007", "2026-08-11", "Rotated our live key but the old one still works", "We rotated our live secret key from the dashboard this morning as part of our quarterly rotation. Two hours later, requests using the old key still succeed. Is rotation delayed, or did it not take? We expected the old key to stop working straight away. Also, is there an audit log showing which key made which request?"),
    ("T-1008", "2026-08-12", "How do we simulate a dispute in test mode?", "We're building our dispute.opened handler and can't find a way to trigger a dispute against a test-mode charge. Is there a test card number or a dashboard button that opens a dispute? Also, what fields does the dispute object carry? Not urgent, we're a few weeks from launch."),
    ("T-1009", "2026-08-13", "Batch import script times out around row 200", "We're migrating 1,800 customers from our old provider. The script imports each saved payment method and immediately creates the first invoice charge. It runs fine for about 200 rows and then requests start timing out at 30 seconds; the script retries each failed row up to three times and eventually finishes, but the whole run takes hours. Is there a batch endpoint, or a recommended concurrency for imports? Somewhat related: our finance team says a dozen or so of the imported customers have two invoice charges instead of one, which we think is the retry path. We can clean that up ourselves once the import is stable."),
    ("T-1010", "2026-08-14", "Duplicate webhook deliveries and events arriving out of order", "We're seeing the same evt_ id delivered two or three times, sometimes minutes apart, and occasionally a refund.succeeded arrives before the charge.succeeded for the same order. No money problem that we can see, every charge is correct in the dashboard, but our handler assumed one delivery per event and now double-writes some rows. Is this expected behaviour? Should we be deduplicating on our side and, if so, on which field?"),
    ("T-1011", "2026-08-17", "Intermittent 502s from POST /v1/charges since Tuesday", "Roughly one in fifty POST /v1/charges calls returns a 502 with an HTML body rather than your JSON error format, all from the eu-west region. Our client retries 5xx with backoff as your errors page recommends and the retry almost always succeeds, so checkout is mostly working. Request ids for six failures are attached. Two customers have emailed our support saying their statement shows the same order twice, which I assume is the retry; we'll sort that out with them. The 502s are what we need you to look at."),
    ("T-1012", "2026-08-18", "Do you support Apple Pay and Google Pay?", "Our product team wants wallet payments at checkout. I can't find Apple Pay or Google Pay anywhere in the docs. Is it supported through Halcyon.js, on a roadmap, or would we need a second provider for wallets? Also, is there a fee difference for wallet payments?"),
]]

def corpus_fingerprint(docs, tickets):
    return hashlib.sha256(json.dumps([docs, tickets], sort_keys=True).encode()).hexdigest()[:12]

SUMMARIZE = ("You write one-line summaries for Halcyon's support triage queue. Summarize the ticket below "
             "in one sentence of at most 25 words, in plain English.\n\nTicket:\n{ticket}")

def ticket_text(ticket):
    return f"Subject: {ticket['subject']}\n\n{ticket['body']}"

class AppState(TypedDict):
    question: str
    context: list[str] | None
    answer: str

SYSTEM = ("You are Halcyon's developer support assistant. Halcyon is a payments API vendor. When "
          "documentation pages are supplied, answer from them alone and say so if they do not cover the "
          "question. Be concise and specific.")

def respond(state: AppState) -> dict:
    prompt = state["question"]
    if state.get("context"):
        prompt = "Documentation pages:\n\n" + "\n\n".join(state["context"]) + f"\n\nQuestion: {prompt}"
    reply = with_retry(llm.invoke, [SystemMessage(SYSTEM), HumanMessage(prompt)])
    USAGE.append(reply.usage_metadata or {})
    return {"answer": reply.text.strip()}

builder = StateGraph(AppState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)
APP = builder.compile()

def answer(question, context=None):
    return APP.invoke({"question": question, "context": context})["answer"]

SLA = next(d for d in DOCS if d["slug"] == "support-sla")["body"]
CLASSIFY = ("Assign a support priority to the ticket summary below using Halcyon's SLA. Reply with exactly "
            "one of P1, P2 or P3 and nothing else.\n\nSLA:\n{sla}\n\nTicket summary: {summary}")

def triage(ticket):
    """Ticket in, one-sentence summary plus severity out. The severity step sees only the summary."""
    summary = answer(SUMMARIZE.format(ticket=ticket_text(ticket)))
    verdict = re.search(r"P[123]", answer(CLASSIFY.format(sla=SLA, summary=summary)))
    return {"id": ticket["id"], "summary": summary, "severity": verdict.group(0) if verdict else "unparsed"}

GOLDEN_SPEC = {  # severity, trap, pages, reference summary, why it exists
    "T-1001": ("P2", "two-truths", ["webhook-signatures-v1", "webhook-signatures-v2", "changelog-2026-03", "api-conventions"], "Live webhooks carry only the v1 X-Halcyon-Signature header, so SDK 3.1 verification fails; asks whether to move the account to API version 2026-03-01.", "Both signature pages are published and both are correct for someone; the right answer depends on the account's API version."),
    "T-1002": ("P3", None, ["rate-limits"], "CI hits test-mode rate limits at 40 requests per second; asks whether the limit is documented or can be raised.", "A plain how-to with one page that answers it fully; the control case."),
    "T-1003": ("P1", "money-buried", ["webhooks-overview", "api-conventions"], "Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports several orders charged twice during the incident.", "The ticket is about retries; the double charge is an aside at the end. P1 under the SLA."),
    "T-1004": ("P2", None, ["api-conventions", "changelog-2026-03", "errors"], "Retries reusing an Idempotency-Key fail with 400 idempotency_key_reused on live traffic because the retry body differs; asks for the correct retry pattern.", "Needs two pages combined: the conventions rule and the changelog entry that changed it."),
    "T-1005": ("P3", None, ["refunds", "charges-create"], "Second partial refund rejected with amount_exceeds_refundable; asks how to read the remaining refundable balance before refunding.", "Mentions refunds and amounts without any funds being affected; a P3 that sounds like money."),
    "T-1006": ("P1", "money-buried", ["changelog-2026-03", "webhook-signatures-v2", "webhook-signatures-v1"], "SDK 3.0 upgrade made webhook verification fail and was rolled back; two customers were charged twice during the stall; asks the migration order.", "Leads with a failed upgrade; the double charge is a side effect the developer plays down."),
    "T-1007": ("P3", None, ["authentication"], "After a scheduled key rotation the old live key still works two hours later; asks whether that is expected and for how long.", "Sounds like a security incident and is documented behaviour; tests calm reading of docs."),
    "T-1008": ("P3", "no-coverage", [], "Cannot trigger a dispute against a test-mode charge; asks how to simulate one and what the dispute object contains.", "No page covers this. The correct answer is to say so, not to invent a test card."),
    "T-1009": ("P1", "money-buried", ["rate-limits", "api-conventions", "charges-create"], "Bulk import of 1,800 customers times out after about 200 rows; retries have created duplicate invoice charges for a dozen customers.", "Leads with timeouts and batch advice; the duplicate charges are 'somewhat related'."),
    "T-1010": ("P3", "near-miss", ["webhooks-overview"], "Same webhook event delivered several times and out of order; no money affected; asks whether that is expected and how to deduplicate.", "Uses 'duplicate' and 'double' about deliveries, not charges. A keyword rule fires; the SLA does not."),
    "T-1011": ("P1", "money-buried", ["errors", "charges-create", "api-conventions"], "About one in fifty live POST /v1/charges calls returns 502 from eu-west; retries have charged two customers twice.", "Leads with 502s and follows the errors page's own retry advice; the retry is what double-charged customers."),
    "T-1012": ("P3", "no-coverage", [], "Asks whether Apple Pay and Google Pay are supported through Halcyon.js or planned.", "No page covers wallets. Tests whether the assistant admits a documentation gap."),
}

GOLDENS = []
for t in TICKETS:
    severity, trap, pages, summary, reason = GOLDEN_SPEC[t["id"]]
    GOLDENS.append(Golden(name=t["id"], input=ticket_text(t), expected_output=summary,
                          additional_metadata={"expected_severity": severity, "trap": trap,
                                               "pages": pages, "reason": reason}))

CORPUS_FINGERPRINT = corpus_fingerprint(DOCS, TICKETS)
assert CORPUS_FINGERPRINT == "e4485ba0f187", "DOCS or TICKETS drifted from post 0"
print("corpus fingerprint", CORPUS_FINGERPRINT)

corpus fingerprint e4485ba0f187


The fingerprint matched: `e4485ba0f187`. The next cell is carried forward from the earlier metric posts. It builds the judge, gemini-3.6-flash, and wraps the client so every judge call lands in our own ledger with its thinking tokens. Those are the tokens the model spends reasoning before it answers, and Google bills them as output. DeepEval's counter does not see them. We measure the gap at the end. It also prints the temperature DeepEval will send, 0.0, which the earlier posts found buys no determinism on this model family. We leave it as it is.

In [2]:
# Carried forward from the earlier metric posts. Everything above is the shared setup cell, unchanged.
import asyncio
import google.genai.models as genai_models
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, SummarizationMetric
from deepeval.models import GeminiModel
from deepeval.models.llms.constants import GEMINI_MODELS_DATA
from deepeval.test_case import LLMTestCase

MONEY = re.compile(r"twice|double|duplicat|second (charge|time)|two (charge|invoice)|charged again|"
                   r"extra charge|multiple charge|overcharg|billed (again|twice)", re.I)   # the trap check
MONEY_TICKET_IDS = ("T-1003", "T-1006", "T-1009", "T-1011")

JUDGE_USAGE = []   # (usage_metadata, temperature sent) for every judge call; DeepEval's counter omits thinking
_generate_content = genai_models.AsyncModels.generate_content

async def counted_generate_content(self, *args, **kwargs):
    response = await _generate_content(self, *args, **kwargs)
    JUDGE_USAGE.append((response.usage_metadata, kwargs["config"].temperature))
    return response

genai_models.AsyncModels.generate_content = counted_generate_content

def judge_calls():
    return len(JUDGE_USAGE)

registry_price = GEMINI_MODELS_DATA[JUDGE_MODEL].input_price, GEMINI_MODELS_DATA[JUDGE_MODEL].output_price
# Same key variable as the app model: GeminiModel would otherwise look for GOOGLE_API_KEY.
judge = GeminiModel(model=JUDGE_MODEL, api_key=os.environ["GEMINI_API_KEY"],
                    cost_per_input_token=PRICE_IN, cost_per_output_token=PRICE_OUT)

async def _measure(metric, case, gate):
    async with gate:
        await metric.a_measure(case, _show_indicator=False)   # no progress spinner in a notebook
    return metric

async def score(metric_cls, cases, **kwargs):
    """One fresh metric per case, MAX_CONCURRENCY cases in flight; returns {case name: measured metric}."""
    gate = asyncio.Semaphore(MAX_CONCURRENCY)
    metrics = [metric_cls(model=judge, **kwargs) for _ in cases]
    await asyncio.gather(*(_measure(m, c, gate) for m, c in zip(metrics, cases)))
    return {c.name: m for c, m in zip(cases, metrics)}

print(f"deepeval {version('deepeval')}   judge {judge.get_model_name()}")
print(f"temperature DeepEval will send: {judge.temperature}   "
      f"(its registry says supports_temperature={judge.supports_temperature()})")
print(f"price per 1M tokens we set: ${PRICE_IN * 1e6:.2f} in, ${PRICE_OUT * 1e6:.2f} out   "
      f"registry default: ${registry_price[0] * 1e6:.2f} in, ${registry_price[1] * 1e6:.2f} out")
print(f"app model calls: {len(USAGE)}   judge calls: {judge_calls()}")

deepeval 4.2.1   judge gemini-3.6-flash (Gemini)
temperature DeepEval will send: 0.0   (its registry says supports_temperature=True)
price per 1M tokens we set: $0.75 in, $3.75 out   registry default: $1.50 in, $7.50 out
app model calls: 0   judge calls: 0


## What the previous post paid, and what it bought

The previous post's bill is the spine of this one. So we carry its ledger over as four constants and print them before anything else. Then the data. The twelve tickets are canonical and we never edit them. They already hold most of the truth this post extracts against. The severity is in the goldens. A golden is a hand-labelled example, the input and what the right output should hold. The four tickets that report affected funds are named in the carried cell above. What the twelve do not hold is an account id. No developer wrote one. So the account id check has nothing to catch on the twelve, and we say so rather than pretend.

Four more tickets, written for this post, carry what the twelve lack. Developers who wrote their account id into the text, in three different shapes. And money reports worded to fool a keyword rule, which the last section needs. They sit in their own list, outside the fingerprint, and their truth is ours. The sync job's account column takes `acct_` and twelve lowercase letters or digits. That form is one regular expression, `ACCOUNT_FORM`.

Let's lay the data out.

In [3]:
# New in this post. Everything above is carried forward unchanged.
from pydantic import BaseModel, ConfigDict, ValidationError
from typing import Literal
from deepeval.metrics import BaseMetric, GEval, JsonCorrectnessMetric, PromptAlignmentMetric, ExactMatchMetric, PatternMatchMetric
from deepeval.scorer import Scorer
from deepeval.test_case import SingleTurnParams

PREVIOUS = {"judge calls": 500, "cost": 1.0388, "judge calls per case, tree": 3, "verdicts gained": 2, "verdicts": 50}   # the criteria post's ledger

# Four tickets written for this post. They carry what the twelve do not: account ids in the developer's own hand,
# and money reports worded to fool a keyword rule. They sit outside the fingerprinted set on purpose.
EXTRA_TICKETS = [dict(zip(('id', 'submitted', 'subject', 'body'), row)) for row in [
    ("T-1013", "2026-08-19", "Customer says she was charged twice, statement shows a hold", "One of our customers emailed that her card was charged twice for a single order. We looked it up under our account acct_3f9d8c2b7a1e. The first line on her statement was a pending authorization from an attempt that failed on our side, and it dropped off two days later. The second line is the real charge. So no double charge, just a hold that worried her. Is there a way to void an authorization ourselves rather than waiting for it to expire, and is there standard wording we can give customers about pending holds?"),
    ("T-1014", "2026-08-20", "Refund marked succeeded nine days ago has not reached the customer", "We issued a refund of 45.00 GBP for a customer last Tuesday and the refund object has shown status succeeded since Wednesday. The customer says nothing has appeared on her statement and it is now nine days. We are on API version 2026-03-01, account ACCT-7C4E2A9F0B1D from the dashboard. Is there anything we can do from our side, or a reference number we can give her bank to trace it?"),
    ("T-1015", "2026-08-21", "Checkout retry created a second charge for one order", "Our checkout timed out on a POST /v1/charges call on Friday evening, retried, and the customer ended up charged twice for one order. We have refunded the extra charge already. Our account is acct_9b2e4d7f1c3a. We are not sending an Idempotency-Key header at the moment; is that what would have prevented this, and does it apply to refunds as well?"),
    ("T-1016", "2026-08-24", "Payout for the week of 3 August has not landed", "The payout for the week of 3 August shows as paid in the dashboard but nothing has landed in our bank account, and it is normally there within two days. Our account is acct_5A1C8E2F9B3D and the bank details have not changed. Every other payout this quarter arrived on time. Can you tell us where it is, or what reference to give our bank?"),
]]
ALL_TICKETS = TICKETS + EXTRA_TICKETS

# What the ticketing system's record should hold, per ticket. The twelve come from the goldens; the four are ours.
SEVERITY_TRUTH = {}      # SEVERITY_TRUTH[ticket id] -> P1, P2 or P3
FUNDS_TRUTH = {}         # FUNDS_TRUTH[ticket id] -> True if the ticket reports funds affected
ACCOUNT_TRUTH = {}       # ACCOUNT_TRUTH[ticket id] -> the account id in the form the sync job stores, or None
for golden in GOLDENS:
    SEVERITY_TRUTH[golden.name] = golden.additional_metadata["expected_severity"]
    FUNDS_TRUTH[golden.name] = golden.name in MONEY_TICKET_IDS
    ACCOUNT_TRUTH[golden.name] = None      # no ticket in the fingerprinted set states one
SEVERITY_TRUTH.update({"T-1013": "P3", "T-1014": "P1", "T-1015": "P1", "T-1016": "P1"})
FUNDS_TRUTH.update({"T-1013": False, "T-1014": True, "T-1015": True, "T-1016": True})
ACCOUNT_TRUTH.update({"T-1013": "acct_3f9d8c2b7a1e", "T-1014": "acct_7c4e2a9f0b1d",
                      "T-1015": "acct_9b2e4d7f1c3a", "T-1016": "acct_5a1c8e2f9b3d"})
ACCOUNT_FORM = r"acct_[a-z0-9]{12}"     # the sync job's column: acct_ and twelve lowercase letters or digits

def ticket_input(ticket):
    """The ticket as the extractor sees it: the id and date the ticketing system knows, then the developer's text."""
    return f"Ticket: {ticket['id']}\nSubmitted: {ticket['submitted']}\nSubject: {ticket['subject']}\n\n{ticket['body']}"

funds_true = 0
for ticket_id in FUNDS_TRUTH:
    funds_true += FUNDS_TRUTH[ticket_id]
stated_accounts = 0
for ticket_id in ACCOUNT_TRUTH:
    stated_accounts += ACCOUNT_TRUTH[ticket_id] is not None
print(f"tickets            {len(ALL_TICKETS)}   ({len(TICKETS)} from the fingerprinted set, {len(EXTRA_TICKETS)} written for this post)")
print(f"funds affected     {funds_true} of {len(ALL_TICKETS)}")
print(f"account id stated  {stated_accounts} of {len(ALL_TICKETS)}")
print(f"previous post      {PREVIOUS['judge calls']} judge calls, ${PREVIOUS['cost']:.4f}, tree at {PREVIOUS['judge calls per case, tree']} calls per case, "
      f"{PREVIOUS['verdicts gained']} verdicts gained out of {PREVIOUS['verdicts']}")

tickets            16   (12 from the fingerprinted set, 4 written for this post)
funds affected     7 of 16
account id stated  4 of 16
previous post      500 judge calls, $1.0388, tree at 3 calls per case, 2 verdicts gained out of 50


Sixteen tickets. Twelve from the fingerprinted set and four written here. Seven report affected funds. Four state an account id, and all four are new. The last line is the previous post, as constants. 500 judge calls and $\$1.0388$. The tree at 3 calls per case, and 2 verdicts gained out of 50. Every number below sits against those four.

Now the extractor. It is one LangGraph node, a step in a graph from the library every application in this series is built with. Its whole prompt is a numbered list of rules plus Halcyon's SLA page. That page says which tickets are P1, P2 and P3. The list is a Python list, `FIELD_RULES`, because a later metric wants to read the same rules the model was given. The state carries the ticket, the raw text the model wrote, and the parsed record, or `None` when the text did not parse. Two of the metrics below read the raw text, not the dict.

The record's shape is a Pydantic model, `TicketRecord`. Pydantic is a library that checks a JSON document against a declared shape and refuses anything that does not fit. Seven fields. Three of them may be `null`, because most tickets name no version, endpoint or error. Severity is one of three strings. Funds affected is a boolean. One line matters more than it looks: `extra="forbid"`. Without it, Pydantic ignores an eighth key. With it, an eighth key is a failure.

The account id rule says only that the field is the developer's Halcyon account id. It gives no form. Nobody writes the form of an id into a prompt on day one, because an id looks like a thing that gets copied. We come back to that.

Let's build it and run it on the ticket the series always starts with, T-1003.

In [4]:
FIELD_RULES = [      # one list, two readers: the extractor's prompt is built from it, and the alignment metric reads it
    "Return one JSON object and nothing else: no prose before or after it, no code fences.",
    "Use exactly these seven keys and no others: ticket_id, account_id, api_version, endpoint, error_code, severity, funds_affected.",
    "ticket_id is the ticket's id from the Ticket line, for example T-1001.",
    "account_id is the developer's Halcyon account id.",
    "api_version is the Halcyon API version the developer names, as a date such as 2026-03-01.",
    "endpoint is the Halcyon API endpoint the ticket is about, as method and path, for example POST /v1/charges.",
    "error_code is the Halcyon error code or HTTP status the developer reports, for example rate_limit_error or 502.",
    "severity is P1, P2 or P3 under the SLA.",
    "funds_affected is true if the ticket reports that funds were affected, however briefly, and false otherwise.",
    "Use null for any field the ticket does not state.",
]

class TicketRecord(BaseModel):
    model_config = ConfigDict(extra="forbid")     # an eighth key is a failure; Pydantic's default would ignore it
    ticket_id: str
    account_id: str | None
    api_version: str | None
    endpoint: str | None
    error_code: str | None
    severity: Literal["P1", "P2", "P3"]
    funds_affected: bool

numbered_rules = []
for number, rule in enumerate(FIELD_RULES, start=1):
    numbered_rules.append(f"{number}. {rule}")
EXTRACT_SYSTEM = ("You extract fields from Halcyon support tickets for Halcyon's ticketing system.\n\nRules:\n"
                  + "\n".join(numbered_rules) + f"\n\nSLA:\n{SLA}")

class RecordState(TypedDict):
    ticket: dict
    raw: str               # exactly what the model wrote
    record: dict | None    # the parsed JSON, or None when it did not parse

def extract(state: RecordState) -> dict:
    reply = with_retry(llm.invoke, [SystemMessage(EXTRACT_SYSTEM), HumanMessage(ticket_input(state["ticket"]))])
    USAGE.append(reply.usage_metadata or {})
    raw = reply.text.strip()
    try:
        record = json.loads(raw)
    except json.JSONDecodeError:
        record = None
    return {"raw": raw, "record": record}

record_builder = StateGraph(RecordState)
record_builder.add_node("extract", extract)
record_builder.add_edge(START, "extract")
record_builder.add_edge("extract", END)
EXTRACTOR = record_builder.compile()

def extract_record(ticket):
    return EXTRACTOR.invoke({"ticket": ticket, "raw": "", "record": None})

sample = next(ticket for ticket in ALL_TICKETS if ticket["id"] == "T-1003")
first = extract_record(sample)
print(ticket_input(sample))
print("-" * 78)
print(first["raw"])
print("-" * 78)
print(f"parsed as JSON: {first['record'] is not None}   truth: severity {SEVERITY_TRUTH['T-1003']}, funds affected {FUNDS_TRUTH['T-1003']}")

Ticket: T-1003
Submitted: 2026-08-05
Subject: Retry storm after our webhook endpoint returned 500s

On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since then we've received roughly 4,000 redeliveries, some for events from days ago, and they are still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just absorb them until the 72 hours are up? Probably unrelated, but while going through the logs we found a few orders from that afternoon that were charged twice, two charge ids each; we assume our order service retried the charge call on timeout and we'll refund them on our side. Mainly we need the retries to stop.
------------------------------------------------------------------------------
{"ticket_id": "T-1003", "account_id": null, "api_version": null, "endpoint": null, "error_code": "500", "severity": "P1", "funds_affected": true}
------------------------------------------------------------------------------
parse

The ticket in full, then the record the model wrote. Seven keys and nothing else. `account_id` is `null`, which is right, because the developer never gave one. `severity` is P1 and `funds_affected` is `true`. The truth line agrees. The one-line summarizer from the first post kept losing the double charge in this ticket. The extractor, which reads the whole ticket and is asked a direct question, kept it.

One value is arguable. `error_code` is `500`. The 500s in this ticket came from the developer's own webhook endpoint after their bad deploy. Halcyon returned nothing. Whether that belongs in a column called Halcyon error code is a question of reading, and we come back to it.

Let's run all sixteen and freeze the records. Every metric below scores these same sixteen strings. The extractor runs once.

In [5]:
RECORDS = {}     # RECORDS[ticket id] -> {"raw": what the model wrote, "record": the parsed dict or None}
with ThreadPoolExecutor(MAX_CONCURRENCY) as pool:      # the setup cell's cap on calls in flight, as the first post ran its full set
    results = list(pool.map(extract_record, ALL_TICKETS))
for ticket, result in zip(ALL_TICKETS, results):
    RECORDS[ticket["id"]] = result

print(f"{'ticket':<8}{'parsed':<8}{'severity':<10}{'truth':<7}{'funds':<7}{'truth':<7}{'account_id':<20}{'error_code':<28}{'endpoint'}")
for ticket_id in RECORDS:
    record = RECORDS[ticket_id]["record"]
    parsed = record is not None
    severity = record["severity"] if parsed else "-"
    funds = record["funds_affected"] if parsed else "-"
    account = record["account_id"] if parsed else "-"
    error = record["error_code"] if parsed else "-"
    endpoint = record["endpoint"] if parsed else "-"
    print(f"{ticket_id:<8}{str(parsed):<8}{severity:<10}{SEVERITY_TRUTH[ticket_id]:<7}{str(funds):<7}{str(FUNDS_TRUTH[ticket_id]):<7}"
          f"{str(account):<20}{str(error):<28}{endpoint}")
print(f"\napp calls so far: {len(USAGE)}   judge calls so far: {judge_calls()}")

ticket  parsed  severity  truth  funds  truth  account_id          error_code                  endpoint
T-1001  True    P3        P2     False  False  None                no v2 signature present     None
T-1002  True    P3        P3     False  False  None                rate_limit_error            None
T-1003  True    P1        P1     True   True   None                500                         None
T-1004  True    P2        P2     False  False  None                idempotency_key_reused      None
T-1005  True    P3        P3     False  False  None                amount_exceeds_refundable   None
T-1006  True    P1        P1     True   True   None                no v2 signature present     None
T-1007  True    P2        P3     False  False  None                None                        None
T-1008  True    P3        P3     False  False  None                None                        None
T-1009  True    P1        P1     True   True   None                None                        N

Sixteen records, all sixteen parsed. The severity column against its truth column first. T-1001 came out P3 and should be P2. A live launch blocked by failing signature verification is a live integration failing. T-1007 came out P2 and should be P3. A key rotation that behaves as documented is a how-to. Fourteen of sixteen severities are right, and the funds column matches its truth on all sixteen.

Now the account id column. Twelve `None`, which is correct. Then the four new tickets. T-1013 and T-1015 wrote their ids in the canonical form and the extractor copied them. T-1014 wrote `ACCT-7C4E2A9F0B1D`, in capitals with a dash, the way a dashboard might display it. T-1016 wrote `acct_5A1C8E2F9B3D`. The extractor copied both exactly as written. Both are strings. Both would load into the schema. Neither would load into the sync job.

## Schema and pattern checks, at zero judge calls

Three of DeepEval's metrics contain no judge. Here is what each one computes.

### JsonCorrectnessMetric
**Plain meaning:** does the text the model wrote load into the shape we declared, with nothing missing, nothing extra and every type right.
**Under the hood:** it calls `TicketRecord.model_validate_json` on the output. If Pydantic accepts the text, the score is 1. If Pydantic raises, the score is 0. That is the whole score, and it costs zero judge calls. With `include_reason` on, which is the default, it makes one judge call to write a reason, and only when the text failed. A valid record gets a fixed sentence and no call.
**Needs:** `input` and `actual_output`. Referenceless.
**Confused with:** PromptAlignment, later in this post, which asks a judge whether the output followed the instruction "return JSON". The tell is what each one reads. JsonCorrectness never sees the prompt. It sees a schema class.
**Use when:** the output feeds a parser.
**Do not use when:** the content of a field is the question. A valid record with the wrong severity scores 1.

### PatternMatchMetric
**Plain meaning:** does the whole output match a regular expression.
**Under the hood:** it compiles the pattern once, strips the output, and calls `re.fullmatch`. A full match scores 1, anything else 0. Zero judge calls. There is no other code path.
**Needs:** `input` and `actual_output`. Referenceless. It reads the whole `actual_output`, so to check one field we hand it that field as the output.
**Confused with:** ExactMatch, next. Exact match checks identity against one known value. Pattern match checks form against a grammar. The tell is whether a value the metric has never seen can pass. Under a pattern it can.
**Use when:** a field has a grammar. Ids, dates, enums, header names.
**Do not use when:** a wrong value can have the right form. A well-formed id for the wrong account passes.

### ExactMatchMetric
**Plain meaning:** is the output, ignoring surrounding whitespace, the same string as the value we expected.
**Under the hood:** it strips both strings and compares them with `==`. Equal scores 1, different scores 0. Zero judge calls.
**Needs:** `input`, `actual_output` and `expected_output`. Reference-based, so it needs a labelled truth.
**Confused with:** PatternMatch above, and with `quasi_exact_match_score` in the scorer module, which folds case and punctuation before comparing. The tell is `P1` against `p1`. Exact match fails it, quasi passes it.
**Use when:** the right value is known and has one spelling. Severity against a golden.
**Do not use when:** the field is prose. A summary with one word moved scores 0.

The test case for a whole-record metric is the ticket as input and the raw JSON as output. The two string metrics get one field each, rendered as text, with JSON `null` becoming the word `null`. That is why the account pattern reads `acct_[a-z0-9]{12}|null`: a well-formed id, or no id at all, and nothing in between.

Let's run the schema check on all sixteen, then on two records we break on purpose.

In [6]:
RECORD_CASES = {}     # RECORD_CASES[ticket id] -> the whole record as one test case; JSON, alignment and the composed metric read it
for ticket in ALL_TICKETS:
    RECORD_CASES[ticket["id"]] = LLMTestCase(name=ticket["id"], input=ticket_input(ticket), actual_output=RECORDS[ticket["id"]]["raw"])

before = judge_calls()
schema_scores = {}    # schema_scores[ticket id] -> 1.0 if the raw text loads into TicketRecord, else 0.0
for ticket_id in RECORD_CASES:
    metric = JsonCorrectnessMetric(expected_schema=TicketRecord, model=judge)
    await metric.a_measure(RECORD_CASES[ticket_id], _show_indicator=False)
    schema_scores[ticket_id] = metric.score
valid = 0
for ticket_id in schema_scores:
    valid += schema_scores[ticket_id] == 1
print(f"schema valid: {valid} of {len(schema_scores)}   judge calls used: {judge_calls() - before}")
print(f"reason on a valid record: {metric.reason}")

# The same record with the two failures teams meet first: a code fence around it, and an extra key inside it.
fenced = LLMTestCase(input=RECORD_CASES["T-1003"].input, actual_output="```json\n" + RECORDS["T-1003"]["raw"] + "\n```")
with_extra_key = dict(RECORDS["T-1003"]["record"])
with_extra_key["notes"] = "developer will refund"
extra_key = LLMTestCase(input=RECORD_CASES["T-1003"].input, actual_output=json.dumps(with_extra_key))
for label, case in (("fenced", fenced), ("extra key", extra_key)):
    before = judge_calls()
    metric = JsonCorrectnessMetric(expected_schema=TicketRecord, model=judge)
    await metric.a_measure(case, _show_indicator=False)
    print(f"\n{label}: score {metric.score}   judge calls used: {judge_calls() - before}")
    print(f"  reason: {metric.reason}")

schema valid: 16 of 16   judge calls used: 0
reason on a valid record: The generated Json matches and is syntactically correct to the expected schema.



fenced: score 0   judge calls used: 1
  reason: The generated Json is valid because all required fields ('ticket_id', 'account_id', 'api_version', 'endpoint', 'error_code', 'severity', 'funds_affected') are present and conform strictly to the types, nullability, and enum constraints specified in the Expected Json Schema.



extra key: score 0   judge calls used: 1
  reason: The generated JSON is invalid because it contains an extra field 'notes' which is not defined in the schema properties when 'additionalProperties' is set to false.


Sixteen of sixteen valid, and the judge was called zero times. The reason on a valid record is DeepEval's fixed sentence. No model wrote it.

Then two failures we made ourselves. For the first we took the T-1003 record and put a code fence around it. A code fence is the three backticks a model adds when it formats an answer as a code block. Pydantic received the whole string, backticks included. Backticks are not JSON, so parsing failed and the score is 0. That number came from Pydantic alone. Then, because the score was 0, the metric made its one judge call and asked the judge to explain the failure. The judge read the text the way a person would and skipped the backticks. It found seven fields of the right types and wrote that the JSON is valid. So the metric returned a score of 0 and a reason that says the record is fine. The score is right. The reason is wrong.

The second failure adds an eighth key, `notes`. Score 0, one judge call, and this time the reason is right. It names the key and the `additionalProperties` rule. Without `extra="forbid"` this record would have scored 1.

So the schema metric's judge is a narrator, not a scorer. The score came from Pydantic, and when the narration is wrong the number is still right.

Let's score the two fields that have a grammar and a truth.

In [7]:
def field_text(record, key):
    """The one field as the text a string metric reads; JSON null becomes the word null."""
    value = record[key]
    if value is None:
        return "null"
    return str(value)

ACCOUNT_CASES = {}    # ACCOUNT_CASES[ticket id] -> the account_id field alone
SEVERITY_CASES = {}   # SEVERITY_CASES[ticket id] -> the severity field alone, with the truth as expected_output
for ticket in ALL_TICKETS:
    record = RECORDS[ticket["id"]]["record"]
    ACCOUNT_CASES[ticket["id"]] = LLMTestCase(name=ticket["id"], input=ticket["id"], actual_output=field_text(record, "account_id"))
    SEVERITY_CASES[ticket["id"]] = LLMTestCase(name=ticket["id"], input=ticket["id"], actual_output=field_text(record, "severity"),
                                               expected_output=SEVERITY_TRUTH[ticket["id"]])

before = judge_calls()
account_scores = {}
severity_scores = {}
for ticket_id in ACCOUNT_CASES:
    account_metric = PatternMatchMetric(pattern=ACCOUNT_FORM + "|null")     # a well-formed id, or no id at all
    account_metric.measure(ACCOUNT_CASES[ticket_id], _show_indicator=False)
    account_scores[ticket_id] = account_metric.score
    severity_metric = ExactMatchMetric()
    severity_metric.measure(SEVERITY_CASES[ticket_id], _show_indicator=False)
    severity_scores[ticket_id] = severity_metric.score

print(f"{'ticket':<8}{'schema':<8}{'account id':<20}{'pattern':<9}{'severity':<10}{'truth':<7}{'exact'}")
for ticket_id in RECORDS:
    account = ACCOUNT_CASES[ticket_id].actual_output
    severity = SEVERITY_CASES[ticket_id].actual_output
    print(f"{ticket_id:<8}{schema_scores[ticket_id]:<8.1f}{account:<20}{account_scores[ticket_id]:<9.1f}{severity:<10}{SEVERITY_TRUTH[ticket_id]:<7}{severity_scores[ticket_id]:.1f}")
account_pass = 0
severity_pass = 0
for ticket_id in RECORDS:
    account_pass += account_scores[ticket_id] == 1
    severity_pass += severity_scores[ticket_id] == 1
print(f"\nschema valid {valid} of 16, account id well-formed {account_pass} of 16, severity matches truth {severity_pass} of 16")
print(f"judge calls used by the two string metrics: {judge_calls() - before}")

ticket  schema  account id          pattern  severity  truth  exact
T-1001  1.0     null                1.0      P3        P2     0.0
T-1002  1.0     null                1.0      P3        P3     1.0
T-1003  1.0     null                1.0      P1        P1     1.0
T-1004  1.0     null                1.0      P2        P2     1.0
T-1005  1.0     null                1.0      P3        P3     1.0
T-1006  1.0     null                1.0      P1        P1     1.0
T-1007  1.0     null                1.0      P2        P3     0.0
T-1008  1.0     null                1.0      P3        P3     1.0
T-1009  1.0     null                1.0      P1        P1     1.0
T-1010  1.0     null                1.0      P3        P3     1.0
T-1011  1.0     null                1.0      P1        P1     1.0
T-1012  1.0     null                1.0      P3        P3     1.0
T-1013  1.0     acct_3f9d8c2b7a1e   1.0      P3        P3     1.0
T-1014  1.0     ACCT-7C4E2A9F0B1D   0.0      P1        P1     1.0
T-1015  

The schema column reads 1.0 sixteen times. The pattern column reads 0.0 twice, on T-1014 and T-1016, the two ids copied in capitals. The exact column reads 0.0 twice, on T-1001 and T-1007, the two wrong severities. Sixteen of sixteen valid. Fourteen of sixteen loadable. Fourteen of sixteen right on severity. Zero judge calls across all three checks.

The schema said every record was fine. Two records would have been dropped at three in the morning, and two would have been filed at the wrong priority. Schema validity is the shape of the box. It says nothing about what is in it.

The pattern check found exactly what the story at the top described, and found it for nothing. On the twelve original tickets it found nothing, which is also correct. A gate that says nothing when nothing is wrong is doing its job.

Two more scoring functions belong in this section, because teams reach for them here. DeepEval's scorer module has BLEU and ROUGE. Both come from translation and summarization research. BLEU counts how many word sequences in the output appear in a reference. ROUGE counts the reverse, how much of the reference the output covers. Both give partial credit by design. That is right for a sentence and wrong for an identifier. An id that is one character off is a different account, with no partial credit due.

Let's put both on an identifier and on a summary.

In [8]:
import nltk, warnings
nltk.download("punkt_tab", quiet=True)      # the tokenizer BLEU uses; a one-time download
warnings.filterwarnings("ignore", category=UserWarning, module="nltk")   # BLEU warns about empty 3-gram and 4-gram overlaps

right_id = "acct_7c4e2a9f0b1d"
wrong_id = "acct_7c4e2a9f0b1e"               # one character wrong; the sync job stores it against the wrong account, or no account
reference_summary = GOLDEN_SPEC["T-1003"][3]
model_summary = answer(SUMMARIZE.format(ticket=ticket_text(sample)))

print(f"identifier, one character wrong")
print(f"  exact match  {Scorer.exact_match_score(right_id, wrong_id)}")
print(f"  ROUGE-L      {Scorer.rouge_score(right_id, wrong_id, 'rougeL'):.2f}")
print(f"  BLEU-1       {Scorer.sentence_bleu_score(right_id, wrong_id, 'bleu1'):.2f}")
print(f"\nsummary against the reference summary for T-1003")
print(f"  model:     {model_summary}")
print(f"  reference: {reference_summary}")
print(f"  exact match  {Scorer.exact_match_score(reference_summary, model_summary)}")
print(f"  ROUGE-L      {Scorer.rouge_score(reference_summary, model_summary, 'rougeL'):.2f}")
print(f"  BLEU-1       {Scorer.sentence_bleu_score(reference_summary, model_summary, 'bleu1'):.2f}")
print(f"  money kept   {MONEY.search(model_summary) is not None}")

identifier, one character wrong
  exact match  0
  ROUGE-L      0.50
  BLEU-1       0.00

summary against the reference summary for T-1003
  model:     The customer is asking how to cancel pending webhook retries after a temporary endpoint outage triggered a surge of redeliveries.
  reference: Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports several orders charged twice during the incident.
  exact match  0
  ROUGE-L      0.21
  BLEU-1       0.24
  money kept   False


One character wrong in an account id. Exact match 0. ROUGE-L 0.50, because its tokenizer splits on the underscore. The id becomes two tokens, `acct` and the twelve characters, and the shared prefix matched. BLEU-1 0.00, because NLTK's tokenizer keeps the underscore, so the id is one token and it did not match. The same string scores half marks or none depending on which library reads the underscore. Neither number means anything. The sync job does not store half an account.

On a summary the picture changes. The model's one-line summary of T-1003 against the reference scores ROUGE-L 0.21 and BLEU-1 0.24. Each number is a ratio of shared tokens to total tokens, and each library counts differently. DeepEval's ROUGE wrapper switches on a stemmer, so `retry` and `retries` count as the same word. It credits only words that appear in the same order in both sentences. BLEU keeps case and stems nothing, so `Webhook` and `webhook` are different words. Its tokenizer also counts the final period as a word, and that one matched. Both scores are low, which is fair for two sentences that say the same thing in different words. Neither noticed that the summary dropped the double charge. The money check on the last line did. ROUGE tells us the two sentences share few words. It cannot tell us which words mattered.

So the scorer module has a place, and it is not extraction. For summaries it is a cheap first pass. For fields the check is exact match or pattern. For prose that must contain one specific fact, it is a targeted pattern like `MONEY`, or a judge.

## Instruction compliance with PromptAlignment

The extractor was given ten rules. A natural question is whether it followed them. DeepEval has a metric for exactly that question, and it charges for the answer.

### PromptAlignmentMetric
**Plain meaning:** did the output follow each instruction in the prompt.
**Under the hood:** one judge call receives the instruction list, the input and the output. It returns a yes or no per instruction, with a reason for each no. The score is the count of instructions not marked no, divided by the count of instructions. Ten rules and one no is 0.90. A second judge call writes the overall reason. Two judge calls per test case, whatever the outcome.
**Needs:** `input` and `actual_output`, plus the instruction list at construction. Referenceless.
**Confused with:** G-Eval with steps, from the previous post, which scores a criterion we write as a list of steps. G-Eval scores one criterion of degree. PromptAlignment counts instructions. The tell is the denominator. A PromptAlignment score is always a fraction over the instruction count.
**Use when:** an instruction has no regular expression. "Answer in the developer's own terms." "Do not apologize."
**Do not use when:** the instruction is a regex or a schema. The judge charges two calls to check what `re.fullmatch` checks for nothing, and it can invent a violation.

We hand it `FIELD_RULES`, the same list the extractor was built from, and a threshold of 1.0. A record that broke one rule out of ten is not nine tenths loadable.

Let's run it on T-1014, the ticket with the capitalized account id.

In [9]:
case = RECORD_CASES["T-1014"]
before = judge_calls()
alignment = PromptAlignmentMetric(prompt_instructions=FIELD_RULES, model=judge, threshold=1.0)
await alignment.a_measure(case, _show_indicator=False)
print(f"T-1014 record: {case.actual_output}")
print(f"\nscore {alignment.score:.2f}   judge calls used: {judge_calls() - before}")
for number, verdict in enumerate(alignment.verdicts, start=1):
    print(f"  rule {number:>2}  {verdict.verdict:<4} {verdict.reason or ''}")
print(f"\nreason: {alignment.reason}")

T-1014 record: {"ticket_id": "T-1014", "account_id": "ACCT-7C4E2A9F0B1D", "api_version": "2026-03-01", "endpoint": null, "error_code": null, "severity": "P1", "funds_affected": true}

score 0.90   judge calls used: 2
  rule  1  yes  
  rule  2  yes  
  rule  3  yes  
  rule  4  yes  
  rule  5  yes  
  rule  6  yes  
  rule  7  yes  
  rule  8  yes  
  rule  9  yes  
  rule 10  no   The ticket does not state the severity level, so severity should have been set to null.

reason: The score is 0.90 because the LLM accurately extracted most ticket details, but incorrectly inferred a severity level of 'P1' when it was not stated in the input and should have been set to null.


Two judge calls and a score of 0.90. Rule 4, the account id rule, got a yes, and that verdict is right. The rule names no form, so copying `ACCT-7C4E2A9F0B1D` is following the rule. The rule that got a no is rule 10, use `null` for a field the ticket does not state. The judge's reason: the ticket does not state a severity, so severity should have been `null`. The ticket states no severity because no ticket does. Severity is derived from the SLA, and rule 8 says so. The judge read two rules against each other and marked the record down for following one of them.

So the judge got one verdict wrong and one verdict right, and neither helped. The no is not a violation. The yes is correct, and the record still would not load, because PromptAlignment answers one question only: did the output follow the prompt. The sync job's account form is a requirement the prompt never stated, so the judge could not see it. A prompt-compliance metric inherits every blind spot of the prompt it checks. The pattern check has no prompt to inherit from. It asks the sync job's question directly, and its 0.0 sits next to this 0.90.

## One custom metric that composes a gate with a judgement

The previous post built its gate-first shape as a tree, with a judge at every node. Here every gate is code, and the judge is asked one question at the end. DeepEval's `BaseMetric` is the class to subclass. A metric needs `measure`, `a_measure`, and a `__name__` property, and it sets `score`, `reason` and `success` on itself. That is the whole contract.

Our gates run in order. First the schema, through the same `model_validate_json` the schema metric used. Then the account id form, through `re.fullmatch` on `ACCOUNT_FORM`. A record that fails either gate scores 0, gets a reason naming the gate, and never reaches the judge. A record that passes both goes to a G-Eval with three fixed steps we wrote. Passing steps in means one judge call, as the previous post established. The steps ask one thing. Are `api_version`, `endpoint` and `error_code` values the ticket actually states about Halcyon's API. A version the account is on. An endpoint the developer called. An error Halcyon returned. A value the ticket never states, or one that describes the developer's own systems, is unfaithful. A `null` is always fine when the ticket says nothing.

That is the question no pattern can ask, and the only one we pay for.

Let's run it on T-1003, whose error code we flagged as arguable, and on T-1014.

In [10]:
FIDELITY_STEPS = [
    "Read api_version, endpoint and error_code in the record. A value is faithful when the ticket states it as a fact about "
    "Halcyon's API: a version the account is on, an endpoint the developer called, or an error Halcyon returned.",
    "A value the ticket never states, or one that describes the developer's own systems rather than Halcyon's, is unfaithful. "
    "null is always acceptable when the ticket states nothing for that field.",
    "Score 10 if all three values are faithful and 0 if any one is not.",
]

class GatedRecordMetric(BaseMetric):
    """Two code gates, then one judge call. A record that fails a gate scores 0 and never reaches the judge."""

    def __init__(self, model, threshold=0.5):
        self.threshold = threshold
        self.model = model
        self.judged = GEval(name="Field fidelity", evaluation_steps=FIDELITY_STEPS, model=model, threshold=threshold,
                            evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT])

    def gate(self, test_case):
        """The name of the gate that failed, or 'passed'. Zero judge calls: Pydantic and re.fullmatch decide."""
        try:
            record = TicketRecord.model_validate_json(test_case.actual_output)
        except ValidationError:
            return "schema"
        if record.account_id is not None and re.fullmatch(ACCOUNT_FORM, record.account_id) is None:
            return "account id form"
        return "passed"

    def finish(self, score, reason):
        self.score = score
        self.reason = reason
        self.success = self.is_successful()
        return score

    async def a_measure(self, test_case, _show_indicator=False, _in_component=False):
        self.gate_result = self.gate(test_case)
        if self.gate_result != "passed":
            self.evaluation_cost, self.input_tokens, self.output_tokens = 0, 0, 0
            return self.finish(0.0, f"failed the {self.gate_result} gate; not judged")
        await self.judged.a_measure(test_case, _show_indicator=False)
        self.evaluation_cost, self.input_tokens, self.output_tokens = self.judged.evaluation_cost, self.judged.input_tokens, self.judged.output_tokens
        return self.finish(self.judged.score, self.judged.reason)

    def measure(self, test_case, _show_indicator=False, _in_component=False):
        self.gate_result = self.gate(test_case)
        if self.gate_result != "passed":
            self.evaluation_cost, self.input_tokens, self.output_tokens = 0, 0, 0
            return self.finish(0.0, f"failed the {self.gate_result} gate; not judged")
        self.judged.measure(test_case, _show_indicator=False)
        self.evaluation_cost, self.input_tokens, self.output_tokens = self.judged.evaluation_cost, self.judged.input_tokens, self.judged.output_tokens
        return self.finish(self.judged.score, self.judged.reason)

    @property
    def __name__(self):
        return "Gated Record"

for ticket_id in ("T-1003", "T-1014"):
    before = judge_calls()
    gated = GatedRecordMetric(model=judge)
    await gated.a_measure(RECORD_CASES[ticket_id])
    print(f"{ticket_id}  gate: {gated.gate_result:<16} score {gated.score:.2f}   judge calls used: {judge_calls() - before}")
    print(f"        {gated.reason}\n")

T-1003  gate: passed           score 0.00   judge calls used: 1
        The error_code is set to "500", which describes an HTTP status code returned by the developer's own webhook endpoint rather than an error returned by Halcyon's API. Because this value describes the developer's system, it is unfaithful under the evaluation rules.

T-1014  gate: account id form  score 0.00   judge calls used: 0
        failed the account id form gate; not judged



T-1003 passed both gates and reached the judge. One judge call, score 0.00. The reason says the 500 was returned by the developer's own webhook endpoint, not by Halcyon's API. That is the reading we had in the first section. The judge agrees with it, this time.

T-1014 failed the account id gate. Zero judge calls. Score 0.00, and the reason names the gate. The judge never saw the record, and did not need to.

Now the measurement this post exists for. Five metrics, three runs each, over the same sixteen frozen records. The three code metrics, PromptAlignment, and our gated metric. For each we count judge calls from our ledger and cost them at the prices we set. Then we count the tickets whose three scores were the same number. The judged metrics go through the carried `score()` helper, four cases in flight, so three runs fit in a few minutes. The counting is still one metric at a time.

Let's run every metric three times.

In [11]:
RUNS = (1, 2, 3)
METRIC_CLASS = {"JsonCorrectness": JsonCorrectnessMetric, "PatternMatch": PatternMatchMetric, "ExactMatch": ExactMatchMetric,
                "PromptAlignment": PromptAlignmentMetric, "GatedRecord": GatedRecordMetric}
METRIC_ARGS = {"JsonCorrectness": {"expected_schema": TicketRecord}, "PatternMatch": {"pattern": ACCOUNT_FORM + "|null"},
               "ExactMatch": {}, "PromptAlignment": {"prompt_instructions": FIELD_RULES, "threshold": 1.0}, "GatedRecord": {}}
METRIC_CASES = {"JsonCorrectness": RECORD_CASES, "PatternMatch": ACCOUNT_CASES, "ExactMatch": SEVERITY_CASES,
                "PromptAlignment": RECORD_CASES, "GatedRecord": RECORD_CASES}
JUDGED = ("JsonCorrectness", "PromptAlignment", "GatedRecord")     # these take a model; the other two have no model argument

PANEL = {}       # PANEL[metric][run][ticket id] -> the measured metric object
CALL_SPAN = {}   # CALL_SPAN[metric] -> (first judge call index, last) for this metric's three runs, for the ledger
panel_start = time.time()
for name in METRIC_CLASS:
    PANEL[name] = {}
    first_call = judge_calls()
    for run in RUNS:
        cases = list(METRIC_CASES[name].values())
        if name in JUDGED:
            # The judged metrics go through the carried score() helper, MAX_CONCURRENCY cases in flight, so three
            # runs fit the wall-clock budget. Each metric's calls are still counted on their own, one metric at a time.
            PANEL[name][run] = await score(METRIC_CLASS[name], cases, **METRIC_ARGS[name])
        else:
            PANEL[name][run] = {}
            for case in cases:
                metric = METRIC_CLASS[name](**METRIC_ARGS[name])
                metric.measure(case, _show_indicator=False)
                PANEL[name][run][case.name] = metric
    CALL_SPAN[name] = (first_call, judge_calls())
print(f"panel: {len(METRIC_CLASS)} metrics x {len(RUNS)} runs x {len(ALL_TICKETS)} cases, {judge_calls()} judge calls so far, {time.time() - panel_start:.0f} s")

print(f"\n{'metric':<17}{'judge calls':>12}{'per case':>10}{'cost':>10}{'identical':>11}")
COST = {}        # COST[metric] -> dollars over the three runs, from our ledger
IDENTICAL = {}   # IDENTICAL[metric] -> tickets whose three scores were the same number
for name in PANEL:
    first_call, last_call = CALL_SPAN[name]
    tokens_in = 0
    tokens_out = 0
    for usage, temperature in JUDGE_USAGE[first_call:last_call]:
        tokens_in += usage.prompt_token_count
        tokens_out += usage.candidates_token_count + (usage.thoughts_token_count or 0)
    COST[name] = tokens_in * PRICE_IN + tokens_out * PRICE_OUT
    identical = 0
    for ticket_id in METRIC_CASES[name]:
        scores = []
        for run in RUNS:
            scores.append(round(PANEL[name][run][ticket_id].score, 2))
        identical += len(set(scores)) == 1
    IDENTICAL[name] = identical
    calls = last_call - first_call
    print(f"{name:<17}{calls:>12}{calls / (len(RUNS) * len(ALL_TICKETS)):>10.2f}{COST[name]:>10.4f}{identical:>8} of 16")

panel: 5 metrics x 3 runs x 16 cases, 143 judge calls so far, 225 s

metric            judge calls  per case      cost  identical
JsonCorrectness             0      0.00    0.0000      16 of 16
PatternMatch                0      0.00    0.0000      16 of 16
ExactMatch                  0      0.00    0.0000      16 of 16
PromptAlignment            96      2.00    0.4229       4 of 16
GatedRecord                42      0.88    0.0942      14 of 16


One hundred and forty-three judge calls so far, and 225 seconds for the panel. The table reads by row.

JsonCorrectness, PatternMatch and ExactMatch: zero calls, zero dollars, identical on 16 of 16. That last number is a property rather than a finding. `re.fullmatch` has no second behaviour available.

PromptAlignment: 96 calls, 2.00 per case, $\$0.4229$, identical on 4 of 16. Twelve records scored differently on at least one run.

GatedRecord: 42 calls, 0.88 per case, $\$0.0942$, identical on 14 of 16. The 0.88 is 14 records judged out of 16, because two stopped at the gate. Forty-two calls against ninety-six, and about a fifth of the cost, for a metric that asks the harder question.

Let's chart the two columns that matter and list the movers.

In [12]:
print("judge calls per test case, three runs averaged")
for name in PANEL:
    first_call, last_call = CALL_SPAN[name]
    per_case = (last_call - first_call) / (len(RUNS) * len(ALL_TICKETS))
    bar = "#" * round(per_case * 20)
    print(f"  {name:<17}{per_case:>5.2f}  {bar}")
print("\ncost over the panel, dollars")
for name in PANEL:
    bar = "#" * round(COST[name] * 200)
    print(f"  {name:<17}{COST[name]:>8.4f}  {bar}")

# Which cases moved between runs, and what the judge said each time
print("\nscores that were not identical across the three runs")
for name in ("PromptAlignment", "GatedRecord"):
    for ticket_id in RECORD_CASES:
        scores = []
        for run in RUNS:
            scores.append(f"{PANEL[name][run][ticket_id].score:.2f}")
        if len(set(scores)) > 1:
            print(f"  {name:<17}{ticket_id}  {' '.join(scores)}")

judge calls per test case, three runs averaged
  JsonCorrectness   0.00  
  PatternMatch      0.00  
  ExactMatch        0.00  
  PromptAlignment   2.00  ########################################
  GatedRecord       0.88  ##################

cost over the panel, dollars
  JsonCorrectness    0.0000  
  PatternMatch       0.0000  
  ExactMatch         0.0000  
  PromptAlignment    0.4229  #####################################################################################
  GatedRecord        0.0942  ###################

scores that were not identical across the three runs
  PromptAlignment  T-1001  1.00 1.00 0.90
  PromptAlignment  T-1002  1.00 0.90 0.90
  PromptAlignment  T-1003  0.90 0.90 1.00
  PromptAlignment  T-1004  0.90 0.90 1.00
  PromptAlignment  T-1005  1.00 1.00 0.90
  PromptAlignment  T-1006  0.90 1.00 1.00
  PromptAlignment  T-1007  0.90 1.00 1.00
  PromptAlignment  T-1011  0.90 0.90 1.00
  PromptAlignment  T-1013  0.90 0.90 1.00
  PromptAlignment  T-1014  1.00 0.90 1.00
  

The bars are the argument. Three metrics at zero, one at two calls per case, one under one. Cost follows the same shape. PromptAlignment at $\$0.4229$ is four and a half times the gated metric's $\$0.0942$. The gated metric asks about content. PromptAlignment asks about format.

Then the movers. Twelve PromptAlignment records moved between 0.90 and 1.00 across the three runs. The direction is one rule flagged or not flagged, never two. Two GatedRecord records moved, T-1001 and T-1006, and both moved the same way: 0.00, then 1.00, then 0.00. The same string, `no v2 signature present`, judged unfaithful twice and faithful once.

Stability is one axis. The previous post measured a metric that agreed with itself on every run while being wrong on two questions every run. Its conclusion was that spread, the highest score minus the lowest across runs, is not accuracy. So the second axis is whether each metric was right, read against what we know is true of these sixteen records.

Let's read both judged metrics against the truth.

In [13]:
# The second axis: right, not just stable. Each metric is read against what we know is true of the frozen records.
print("PromptAlignment: which rules the judge marked not followed, per run, and the first reason it gave")
for run in RUNS:
    flagged = []
    first_reason = ""
    for ticket_id in RECORD_CASES:
        metric = PANEL["PromptAlignment"][run][ticket_id]
        for number, verdict in enumerate(metric.verdicts, start=1):
            if verdict.verdict.strip().lower() == "no":
                flagged.append(f"{ticket_id} rule {number}")
                if not first_reason:
                    first_reason = f"{ticket_id} rule {number}: {verdict.reason}"
    print(f"  run {run}: {flagged if flagged else 'nothing flagged'}")
    print(f"         {first_reason[:230]}")
wrong_severity = []
malformed_account = []
for ticket_id in RECORDS:
    if severity_scores[ticket_id] == 0:
        wrong_severity.append(ticket_id)
    if account_scores[ticket_id] == 0:
        malformed_account.append(ticket_id)
print(f"  records with a wrong severity, rule 8: {wrong_severity}")
print(f"  records with a malformed account id: {malformed_account}   (rule 4 names no form, so copying is compliant)")

# The key for the judged step, written against FIDELITY_STEPS after reading the frozen records. An exception raised by
# Halcyon's SDK on the developer's own server is not an error Halcyon returned, so that string fails the second step.
FIDELITY_KEY = {}    # FIDELITY_KEY[ticket id] -> True when every stated api_version, endpoint and error_code is faithful
for ticket_id in RECORDS:
    record = RECORDS[ticket_id]["record"]
    FIDELITY_KEY[ticket_id] = record["error_code"] not in ("500", "no v2 signature present")

print("\nGatedRecord: gate result, judged score per run, and the key")
print(f"{'ticket':<8}{'gate':<17}{'run 1':>6}{'run 2':>6}{'run 3':>6}{'key':>6}   reason in run 1")
right = {}      # right[run] -> judged verdicts that agree with the key, over the records that reached the judge
for run in RUNS:
    right[run] = 0
judged_count = 0
for ticket_id in RECORD_CASES:
    scores = []
    for run in RUNS:
        scores.append(f"{PANEL['GatedRecord'][run][ticket_id].score:.1f}")
    gate = PANEL["GatedRecord"][1][ticket_id].gate_result
    reason = PANEL["GatedRecord"][1][ticket_id].reason
    key = "-"
    if gate == "passed":
        judged_count += 1
        key = "pass" if FIDELITY_KEY[ticket_id] else "fail"
        for run in RUNS:
            right[run] += PANEL["GatedRecord"][run][ticket_id].success == FIDELITY_KEY[ticket_id]
    print(f"{ticket_id:<8}{gate:<17}{scores[0]:>6}{scores[1]:>6}{scores[2]:>6}{key:>6}   {reason[:80]}")
agreement = []
for run in RUNS:
    agreement.append(f"run {run}: {right[run]} of {judged_count}")
print(f"\njudged verdicts that agree with the key: {', '.join(agreement)}")

PromptAlignment: which rules the judge marked not followed, per run, and the first reason it gave
  run 1: ['T-1003 rule 10', 'T-1004 rule 10', 'T-1006 rule 10', 'T-1007 rule 10', 'T-1009 rule 10', 'T-1011 rule 10', 'T-1013 rule 9', 'T-1015 rule 10']
         T-1003 rule 10: The ticket does not state the severity, but the model assigned 'P1' instead of null.
  run 2: ['T-1002 rule 10', 'T-1003 rule 10', 'T-1004 rule 10', 'T-1009 rule 10', 'T-1011 rule 10', 'T-1013 rule 10', 'T-1014 rule 10']
         T-1002 rule 10: The ticket does not state the severity level, but the output provided 'P3' instead of null.
  run 3: ['T-1001 rule 10', 'T-1002 rule 10', 'T-1005 rule 9', 'T-1009 rule 10', 'T-1015 rule 10', 'T-1016 rule 10']
         T-1001 rule 10: The ticket does not state the severity level (P1, P2, or P3), but the output provided 'P3' instead of null.
  records with a wrong severity, rule 8: ['T-1001', 'T-1007']
  records with a malformed account id: ['T-1014', 'T-1016']   (rule 4 name

PromptAlignment first. Across three runs the judge flagged eight, seven and six records. All but two of those flags are rule 10. Every rule 10 reason reads the same way. The ticket does not state a severity, so the field should be `null`. That is the misreading we saw on T-1014, repeated. The other two flags, once on T-1013 and once on T-1005, are rule 9, funds affected, on records whose funds field matches the truth. Twenty-one flags, zero real violations found. Meanwhile the two records with a wrong severity, T-1001 and T-1007, are a rule 8 violation. Rule 8 was never flagged, on any record, in any run. The two malformed ids comply with rule 4 as written. The judge was right to pass them, and the record still would not load. Ninety-six judge calls bought twenty-one false alarms and no true one.

GatedRecord next. The key is written against the steps. `500` describes the developer's endpoint. `no v2 signature present` is an exception raised by Halcyon's SDK on the developer's server, not an error Halcyon returned. So T-1001, T-1003 and T-1006 fail the key, and the other eleven judged records pass. The judge agreed with the key on 14 of 14 in run one and run three. In run two it passed T-1001 and T-1006, 12 of 14. T-1003 failed on all three runs. Every record that failed a gate did so identically, three times, because a gate has one behaviour.

Put the three shapes in a scenario. The suite runs on every deploy. February passed, March failed, and someone asks what changed. The answer depends on which shape produced the score. For the pattern check, the only thing that can change between February and March is the record itself. `re.fullmatch` has one behaviour, so a new failure means the extractor wrote a different string, and we can show the string. For the gated metric, the gates answer the same way, and the one judged question can flip. When it does, we can point at the string the judge read two ways, and at the reason it gave each time. The question was narrow enough to name. For PromptAlignment, twelve of sixteen records scored differently across three runs on identical input, and the reasons were wrong. When March is different we cannot tell whether the extractor moved or the judge did, and the reasons cannot tell us either. That is the difference between a check that can be audited and one that only produces a number.

## The check that looks deterministic and is not

One field on the record looks like it belongs with the assertions. `funds_affected` is a boolean. The first post wrote a regular expression for it, `MONEY`, matching "twice", "double", "duplicate" and a few more. It found the four money tickets there. It is in the carried cell above. A boolean from a regex costs nothing and never moves. So why did the previous post spend a tree on the P1 rule?

Let's run the keyword rule over all sixteen tickets, next to the truth and next to what the extractor wrote. Then we ask the extractor three more times on the tickets where it could go either way.

In [14]:
keyword_errors = []
extractor_errors = []
print(f"{'ticket':<8}{'truth':<7}{'MONEY':<7}{'extractor':<11}")
for ticket in ALL_TICKETS:
    ticket_id = ticket["id"]
    keyword_says = MONEY.search(ticket["body"]) is not None
    extractor_says = RECORDS[ticket_id]["record"]["funds_affected"]
    if keyword_says != FUNDS_TRUTH[ticket_id]:
        keyword_errors.append(ticket_id)
    if extractor_says != FUNDS_TRUTH[ticket_id]:
        extractor_errors.append(ticket_id)
    print(f"{ticket_id:<8}{str(FUNDS_TRUTH[ticket_id]):<7}{str(keyword_says):<7}{str(extractor_says):<11}")
print(f"\nkeyword rule wrong on {len(keyword_errors)} of 16: {keyword_errors}")
print(f"extractor wrong on {len(extractor_errors)} of 16: {extractor_errors}")

NEAR_MISS_IDS = ("T-1010", "T-1013", "T-1014", "T-1015", "T-1016")
FUNDS_REPEATS = 3
print(f"\nthe extractor's funds_affected on the near misses, {FUNDS_REPEATS} fresh extractions each (truth in brackets)")
near_misses = []
for ticket in ALL_TICKETS:
    if ticket["id"] in NEAR_MISS_IDS:
        near_misses.append(ticket)
with ThreadPoolExecutor(MAX_CONCURRENCY) as pool:
    repeats = list(pool.map(extract_record, near_misses * FUNDS_REPEATS))     # each near miss FUNDS_REPEATS times, four in flight
for ticket in near_misses:
    reads = []
    for result in repeats:
        if result["record"] is None:
            continue
        if result["record"]["ticket_id"] == ticket["id"]:
            reads.append(str(result["record"]["funds_affected"]))
    print(f"  {ticket['id']}  [{FUNDS_TRUTH[ticket['id']]}]  {' '.join(reads)}")

ticket  truth  MONEY  extractor  
T-1001  False  False  False      
T-1002  False  False  False      
T-1003  True   True   True       
T-1004  False  False  False      
T-1005  False  False  False      
T-1006  True   True   True       
T-1007  False  False  False      
T-1008  False  False  False      
T-1009  True   True   True       
T-1010  False  True   False      
T-1011  True   True   True       
T-1012  False  False  False      
T-1013  False  True   False      
T-1014  True   False  True       
T-1015  True   True   True       
T-1016  True   False  True       

keyword rule wrong on 4 of 16: ['T-1010', 'T-1013', 'T-1014', 'T-1016']
extractor wrong on 0 of 16: []

the extractor's funds_affected on the near misses, 3 fresh extractions each (truth in brackets)


  T-1010  [False]  False False False
  T-1013  [False]  True False False
  T-1014  [True]  True True True
  T-1015  [True]  True True True
  T-1016  [True]  True True True


The keyword rule is wrong on 4 of 16. T-1010 says "duplicate" and "double" about webhook deliveries, and no money moved. T-1013 says "charged twice" and then explains that the second line was an authorization hold. The rule fires on both. T-1014 is a refund that has not arrived and T-1016 is a payout that has not landed. Neither uses any word on the list, and both are P1 under the SLA. The rule misses both. Two false alarms, two misses, from a pattern that never varies.

The extractor's frozen records were right on all sixteen. Then we asked it three more times about T-1010 and the four new tickets. Four of the five came back the same three times. T-1013 did not. Once it read the hold ticket as funds affected, twice it did not. The frozen record happened to be right. A judge reading that ticket is right most of the time and the regex is wrong every time, and the judge moves.

That is the sorting rule this post leaves behind. A criterion is deterministic when the thing being checked is a form: a shape, a grammar, a known string. `re.fullmatch` is stable because nothing else is available to it. A criterion is a judgement when the thing being checked is a meaning, and a boolean field does not make it a form. Whether "charged twice, but it was a hold" reports affected funds is a meaning. The previous post existed because that meaning needs a reader, and a reader needs a tree to be held to a rule. This post exists because the other six fields on the record do not.

## What teams get wrong

**Paying a judge to check a regex.** Ten rules went to PromptAlignment. Six of them are a schema or a pattern. One JSON object, seven named keys, an id form, a date form, an endpoint form, `null` for absent. Ninety-six judge calls checked those six, flagged twenty-one violations, and every flag was wrong. The pattern check found the two real format failures for nothing. Any rule that can be written as `re.fullmatch` or a Pydantic field is code. It goes in front of the judge, not into the judge's prompt.

**Treating schema validity as correctness.** Sixteen of sixteen records were valid. Two of them would not load into the sync job and two of them were filed at the wrong priority. The schema check is a gate on shape, and a passing gate says the box is the right shape. The one judge call the schema metric makes is for a reason. On the fenced record that reason said the JSON was valid while the score said it was not. The score is the verdict and the reason is a note.

**Writing a custom metric before trying the tree.** Our `GatedRecordMetric` is the right tool here because both gates are code, and DeepEval's tree nodes are judge calls. The previous post's tree was the right tool there because its gates were judgements. The test before subclassing `BaseMetric` is whether any gate needs a model. If one does, the tree is the better shape, and it prints its path.

**Letting one prompt rule contradict another.** Rule 8 said derive the severity from the SLA. Rule 10 said use `null` for anything the ticket does not state. The extractor resolved that correctly on all sixteen records. The judge did not, nineteen times across three runs. A rule list a judge will read has to be written so that no two rules can be read against each other. The judge will find the reading.

The bill, from our own ledger, next to DeepEval's.

In [15]:
judge_in = 0
judge_visible = 0
judge_thinking = 0
for usage, temperature in JUDGE_USAGE:
    judge_in += usage.prompt_token_count
    judge_visible += usage.candidates_token_count
    judge_thinking += usage.thoughts_token_count or 0
judge_out = judge_visible + judge_thinking
judge_cost = judge_in * PRICE_IN + judge_out * PRICE_OUT

deepeval_cost = 0
deepeval_in = 0
for name in JUDGED:
    for run in RUNS:
        for metric in PANEL[name][run].values():
            deepeval_cost += metric.evaluation_cost or 0
            deepeval_in += metric.input_tokens or 0
panel_in = 0
panel_out = 0
first_call = CALL_SPAN["JsonCorrectness"][0]
last_call = CALL_SPAN["GatedRecord"][1]
for usage, temperature in JUDGE_USAGE[first_call:last_call]:
    panel_in += usage.prompt_token_count
    panel_out += usage.candidates_token_count + (usage.thoughts_token_count or 0)
panel_cost = panel_in * PRICE_IN + panel_out * PRICE_OUT

app_in = 0
app_out = 0
for usage in USAGE:
    app_in += usage.get("input_tokens", 0)
    app_out += usage.get("output_tokens", 0)
app_cost = app_in * PRICE_IN + app_out * PRICE_OUT

per_case = []
for name in PANEL:
    first_call, last_call = CALL_SPAN[name]
    per_case.append(f"{name} {(last_call - first_call) / (len(RUNS) * len(ALL_TICKETS)):.2f}")
print(f"judge calls      {judge_calls()}   (previous post: {PREVIOUS['judge calls']})")
print(f"judge calls per case: {', '.join(per_case)}")
print(f"judge tokens     {judge_in} in / {judge_out} out   (thinking inside 'out': {judge_thinking}, {100 * judge_thinking / judge_out:.0f}%)")
print(f"judge cost       ${judge_cost:.4f}")
print(f"panel alone      {last_call - CALL_SPAN['JsonCorrectness'][0]} calls   ours ${panel_cost:.4f}   DeepEval's counter ${deepeval_cost:.4f}, "
      f"{panel_cost / deepeval_cost:.1f} times less   (input tokens: DeepEval {deepeval_in}, ours {panel_in})")
print(f"app calls        {len(USAGE)}   tokens {app_in} in / {app_out} out   cost ${app_cost:.4f}")
print(f"total            {judge_calls() + len(USAGE)} model calls   ${judge_cost + app_cost:.4f}   {time.time() - T0:.0f} s since the setup cell   (previous post: ${PREVIOUS['cost']:.4f})")

judge calls      143   (previous post: 500)
judge calls per case: JsonCorrectness 0.00, PatternMatch 0.00, ExactMatch 0.00, PromptAlignment 2.00, GatedRecord 0.88
judge tokens     95503 in / 125908 out   (thinking inside 'out': 113243, 90%)
judge cost       $0.5438
panel alone      138 calls   ours $0.5171   DeepEval's counter $0.1149, 4.5 times less   (input tokens: DeepEval 92009, ours 92009)
app calls        33   tokens 18618 in / 33212 out   cost $0.1385
total            176 model calls   $0.6823   330 s since the setup cell   (previous post: $1.0388)


One hundred and forty-three judge calls, against 500 in the previous post. Per case: zero, zero and zero for the three code checks, 2.00 for PromptAlignment, 0.88 for the gated metric. The judge took 95,503 tokens in and returned 125,908, of which 113,243 were thinking, 90 percent. The judge cost $\$0.5438$. For the panel's 138 calls, DeepEval's counter and ours saw the same requests and the same 92,009 input tokens. Ours says $\$0.5171$. DeepEval's says $\$0.1149$, 4.5 times less, because it does not count the thinking. That is the fifth post in a row with that gap. The extractor's 33 calls cost $\$0.1385$. The whole notebook cost $\$0.6823$ and ran 330 seconds from the setup cell. The previous post cost $\$1.0388$, and most of that bought a question this post could not have asked with a pattern.

## What Halcyon has now, and what it still cannot do

The fingerprint matched. The twelve tickets and twelve pages are the ones every earlier post scored, and the four new tickets sit in their own list.

Halcyon now has a third application. The Ticket Field Extractor turns a ticket into a seven-field record, as one LangGraph node with a rule list and a schema. It gets the account id right when a developer writes it in the canonical form, and copies it as written when they do not. Halcyon also has a sorting rule for its criteria, and `GatedRecordMetric` is that rule as a class. The schema gate, the account id pattern and the severity string are code and cost nothing. The one question about meaning sits behind them and costs one call, and none for a record the gates stop.

What Halcyon still cannot do is run any of this without a person. The suite has grown across six posts. Two house rules, a retriever score, a summarizer check, and now a record gate. Every one of them runs when someone opens a notebook and presses run. And everything so far rests on twelve tickets one person wrote, plus four more written for this post. The next post generates a dataset large enough to argue with.

Three rules we take forward. A form is code, a meaning is a judge, and a boolean does not make a meaning into a form. Gates go in front of the judge, in Python when they are assertions and in a tree when they are not. And a stable score is only an answer when the thing producing it has one behaviour available.

### Resources

- DeepEval: JSON correctness: https://deepeval.com/docs/metrics-json-correctness
- DeepEval: Prompt alignment: https://deepeval.com/docs/metrics-prompt-alignment
- DeepEval: Exact match: https://deepeval.com/docs/metrics-exact-match
- DeepEval: Pattern match: https://deepeval.com/docs/metrics-pattern-match
- DeepEval: custom metrics with BaseMetric: https://deepeval.com/docs/metrics-custom
- DeepEval: G-Eval: https://deepeval.com/docs/metrics-llm-evals
- Pydantic: models and `extra="forbid"`: https://docs.pydantic.dev/latest/concepts/models/
- Gemini API pricing: https://ai.google.dev/gemini-api/docs/pricing
- Companion repository with this notebook and its pyproject.toml: [REPO-LINK]